In [1]:
import torch
# from torch import nn
from torch.nn.parameter import Parameter
import torch.nn.functional as F
import torch.optim as optim
import torch.nn.init as init

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

## Task 2

Creating the closed form solution in pytorch

In [2]:
m = 1
mu = 5
k = 500

delta = torch.tensor(mu/(2*m))
w0 = torch.sqrt(torch.tensor(k))
w = torch.sqrt(w0**2 - delta**2)

phi = torch.atan(-delta/w)
A = 1/(2*torch.cos(phi))

def x(t):
    return torch.exp(-delta*t)*(2*A*torch.cos(phi + w*t))

### Task 2.1

Generating training data

In [3]:
n_train = 10

x_train = torch.rand(10) * 0.5  # to get the data points in the interval [0, 0.5]
target_train = torch.tensor([x(t) for t in x_train])    # to get the actual values of x(t)

print(x_train)
print(target_train)

tensor([0.1763, 0.3544, 0.3406, 0.2305, 0.3731, 0.3134, 0.0899, 0.2545, 0.3562,
        0.2120])
tensor([-0.5104,  0.0375,  0.1664,  0.1652, -0.1260,  0.3871, -0.2483,  0.3935,
         0.0215, -0.0670])


In [4]:
simple_model = torch.nn.Sequential(
    torch.nn.Linear(1, 10),
    torch.nn.Tanh(),
    torch.nn.Linear(10, 1)
)

net = simple_model

In [ ]:
optimizer = optim.Adam(net.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()

: 

In [ ]:
# changing the shape of the data
x_train = x_train.reshape(len(x_train),1)
target_train = target_train.reshape(len(target_train),1)

# setting hyperparameters and gettings epoch sizes
batch_size = 10
num_epochs = 200
num_samples_train = len(x_train)
num_batches_train = num_samples_train // batch_size
# num_samples_valid = len(target_val)
# num_batches_valid = num_samples_valid // batch_size

# setting up lists for handling loss/accuracy
train_acc, train_loss = [], []
# valid_acc, valid_loss = [], []
test_acc, test_loss = [], []
cur_loss = 0
losses = []

get_slice = lambda i, size: range(i * size, (i + 1) * size)

for epoch in range(num_epochs):
    # Forward -> Backprob -> Update params
    ## Train
    cur_loss = 0
    net.train()
    for i in range(num_batches_train):
        optimizer.zero_grad()
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        # compute gradients given loss
        target_batch = target_train[slce]
        batch_loss = criterion(output, target_batch)
        batch_loss.backward()
        optimizer.step()
        
        cur_loss += batch_loss.item()
    losses.append(cur_loss / batch_size)

    net.eval()
    ### Evaluate training
    train_preds, train_targs = [], []
    for i in range(num_batches_train):
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])

        preds = output
        # preds = torch.max(output, 1)[1]
        
        train_targs += list(target_train[slce].numpy())
        train_preds += list(preds.data.numpy())
        
    
    if epoch % 10 == 0:
        print("Epoch %2i : Train Loss %f" % (
                epoch+1, losses[-1]))

losses_plot = [loss for loss in losses]
epoch = torch.arange(epoch + 1)
plt.figure()
plt.plot(epoch, losses_plot, 'r')
plt.legend(['Train Loss'])
plt.xlabel('Updates'), plt.ylabel('Loss')

Epoch  1 : Train Loss 0.011577
Epoch 101 : Train Loss 0.007140


(Text(0.5, 0, 'Updates'), Text(0, 0.5, 'Loss'))